# Hiver SDE Intern Take-Home
## Brand Selection Analysis

Goal: select the best brand from the **Customer Support on Twitter** dataset for building an evidence-grounded AI support agent.

We evaluate candidate brands on:
1. Data availability
2. Customer interaction volume
3. Conversation richness
4. Multi-turn support interactions
5. Semantic intent structure
6. Historical response/reuse potential
7. Evaluation richness

**Primary dataset:** Customer Support on Twitter.  
**Secondary dataset:** Banking77 is intentionally not used here because it is optional and restricted to intent work.


In [ ]:
import os
import re
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 60)

SEED = 42
DATA_PATH = "/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv"

print("Configuration ready.")
print("Dataset path:", DATA_PATH)


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded.")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())


## 1. Data Health and Dataset Semantics

The `inbound` field identifies message direction:
- `True` = customer message
- `False` = support/brand message

`author_id` is the actual author, so a customer's `author_id` is not the brand name.


In [ ]:
print("Total rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Unique tweets:", df["tweet_id"].nunique())
print("Unique authors:", df["author_id"].nunique())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nInbound distribution:")
display(df["inbound"].value_counts())


In [ ]:
print("Inbound=True examples:")
display(df[df["inbound"] == True][
    ["tweet_id", "author_id", "inbound", "text"]
].head(5))

print("\nInbound=False examples:")
display(df[df["inbound"] == False][
    ["tweet_id", "author_id", "inbound", "text"]
].head(5))


## 2. Identify Candidate Support Brands

We identify candidate brand/support accounts from accounts producing a substantial number of outbound tweets.

This is only a candidate-generation step; final selection will depend on conversation and intent quality.


In [ ]:
outbound = df[df["inbound"] == False]

brand_candidates = (
    outbound["author_id"]
    .value_counts()
    .rename_axis("brand")
    .reset_index(name="brand_replies")
)

MIN_BRAND_REPLIES = 1000
brands = brand_candidates.loc[
    brand_candidates["brand_replies"] >= MIN_BRAND_REPLIES, "brand"
].tolist()

print("Candidate brands:", len(brands))
display(brand_candidates.head(30))


In [ ]:
# Fast lookups for 2.8M rows
tweet_author = df.set_index("tweet_id")["author_id"].to_dict()
tweet_inbound = df.set_index("tweet_id")["inbound"].to_dict()
tweet_parent = df.set_index("tweet_id")["in_response_to_tweet_id"].to_dict()
tweet_text = df.set_index("tweet_id")["text"].to_dict()

def parse_ids(value):
    if pd.isna(value):
        return []
    return [int(x.strip()) for x in str(value).split(",") if x.strip()]


## 3. Correct Brand ↔ Customer Interaction Mapping

A customer tweet has a customer `author_id`. To associate it with a brand, we inspect brand replies and use `in_response_to_tweet_id` to identify the customer tweet being answered.

This avoids the common mistake of filtering customer rows with `author_id == brand`.


In [ ]:
brand_customer_tweets = defaultdict(set)
brand_customer_pairs = []

brand_outbound = df[
    (df["inbound"] == False) &
    (df["author_id"].isin(brands)) &
    (df["in_response_to_tweet_id"].notna())
][[
    "tweet_id", "author_id", "in_response_to_tweet_id", "text", "created_at"
]]

for row in brand_outbound.itertuples(index=False):
    brand = row.author_id
    parent_id = int(row.in_response_to_tweet_id)
    parent_author = tweet_author.get(parent_id)

    if parent_author is not None and parent_author != brand:
        brand_customer_tweets[brand].add(parent_id)
        brand_customer_pairs.append({
            "brand": brand,
            "customer_tweet_id": parent_id,
            "brand_tweet_id": row.tweet_id
        })

print("Outbound brand replies with parent:", len(brand_outbound))
print("Direct brand/customer response pairs:", len(brand_customer_pairs))


In [ ]:
interaction_stats = []

for brand in brands:
    brand_replies = int(((df["author_id"] == brand) & (df["inbound"] == False)).sum())
    customer_ids = brand_customer_tweets.get(brand, set())

    customer_rows = df[df["tweet_id"].isin(customer_ids)]

    interaction_stats.append({
        "brand": brand,
        "brand_replies": brand_replies,
        "customer_tweets_with_direct_reply": len(customer_rows),
        "unique_customers": customer_rows["author_id"].nunique(),
        "customer_reply_ratio": len(customer_rows) / max(brand_replies, 1)
    })

interaction_stats = (
    pd.DataFrame(interaction_stats)
    .sort_values("customer_tweets_with_direct_reply", ascending=False)
    .reset_index(drop=True)
)

display(interaction_stats.head(30))


## 4. Conversation Graph and Depth

The dataset represents support interactions as a directed graph through `in_response_to_tweet_id`.

We follow parent pointers to estimate the conversation chain and measure multi-turn richness.


In [ ]:
parent_lookup = tweet_parent
author_lookup = tweet_author
inbound_lookup = tweet_inbound

def get_conversation_chain(tweet_id, max_depth=50):
    chain = []
    current = int(tweet_id)
    visited = set()

    for _ in range(max_depth):
        if current in visited:
            break
        visited.add(current)
        chain.append(current)

        parent = parent_lookup.get(current)

        if pd.isna(parent):
            break

        try:
            parent = int(parent)
        except Exception:
            break

        if parent not in parent_lookup:
            break

        current = parent

    return chain[::-1]


In [ ]:
# Candidate brands: top 15 by direct customer interactions
TOP_N = 15
top_interaction_brands = (
    interaction_stats.head(TOP_N)["brand"].tolist()
)

print(top_interaction_brands)

# Test reconstruction on one AppleSupport conversation if present,
# otherwise use the top candidate.
test_brand = "AppleSupport" if "AppleSupport" in brand_customer_tweets else top_interaction_brands[0]
test_customer_id = next(iter(brand_customer_tweets[test_brand]))
chain = get_conversation_chain(test_customer_id)

print("\nTest brand:", test_brand)
print("Conversation length:", len(chain))

preview = pd.DataFrame([
    {
        "tweet_id": tid,
        "author": author_lookup.get(tid),
        "inbound": inbound_lookup.get(tid),
        "text": tweet_text.get(tid)
    }
    for tid in chain
])

display(preview)


In [ ]:
MAX_CONVERSATIONS_PER_BRAND = 10000
conversation_profiles = []

for brand in top_interaction_brands:
    customer_ids = list(brand_customer_tweets.get(brand, set()))

    if len(customer_ids) > MAX_CONVERSATIONS_PER_BRAND:
        rng = np.random.default_rng(SEED)
        customer_ids = rng.choice(
            customer_ids,
            size=MAX_CONVERSATIONS_PER_BRAND,
            replace=False
        ).tolist()

    lengths = []

    for tweet_id in customer_ids:
        chain = get_conversation_chain(tweet_id)
        authors = [author_lookup.get(t) for t in chain]

        if len(chain) >= 2 and brand in authors:
            lengths.append(len(chain))

    if lengths:
        lengths = np.array(lengths)
        conversation_profiles.append({
            "brand": brand,
            "sampled_conversations": len(lengths),
            "mean_turns": lengths.mean(),
            "median_turns": np.median(lengths),
            "2plus_turn_rate": np.mean(lengths >= 2),
            "3plus_turn_rate": np.mean(lengths >= 3),
            "4plus_turn_rate": np.mean(lengths >= 4),
            "6plus_turn_rate": np.mean(lengths >= 6),
            "8plus_turn_rate": np.mean(lengths >= 8)
        })

conversation_profiles = (
    pd.DataFrame(conversation_profiles)
    .sort_values("mean_turns", ascending=False)
    .reset_index(drop=True)
)

display(conversation_profiles)


## 5. Customer Message Samples for Intent Discovery

We use a manageable sample of customer messages per candidate brand.
This stage is for brand selection, not the final intent taxonomy.


In [ ]:
INTENT_SAMPLE_SIZE = 3000
intent_samples = {}

for brand in top_interaction_brands:
    customer_ids = list(brand_customer_tweets.get(brand, set()))

    sample = df[df["tweet_id"].isin(customer_ids)][
        ["tweet_id", "author_id", "text"]
    ].copy()

    sample["text"] = sample["text"].fillna("").astype(str).str.strip()
    sample = sample[sample["text"].str.len() >= 15]

    if len(sample) > INTENT_SAMPLE_SIZE:
        sample = sample.sample(INTENT_SAMPLE_SIZE, random_state=SEED)

    intent_samples[brand] = sample.reset_index(drop=True)

    print(f"{brand:20s} {len(sample):5d} customer messages")


In [ ]:
# Inspect actual customer language before clustering
for brand in [b for b in ["AmazonHelp", "SpotifyCares", "TMobileHelp", "AppleSupport"]
              if b in intent_samples]:
    print("\n" + "=" * 90)
    print(brand)
    print("=" * 90)
    display(
        intent_samples[brand][["text"]].sample(
            min(15, len(intent_samples[brand])),
            random_state=SEED
        )
    )


## 6. TF-IDF + KMeans Intent Structure

The first unsupervised experiment uses TF-IDF + KMeans.

We intentionally do not use HDBSCAN here for brand selection. The previous density-based configuration classified everything as noise, so we use a simple interpretable baseline and compare cluster coherence with silhouette score.


In [ ]:
K_VALUES = [5, 8, 10, 12, 15]

tfidf_matrices = {}
clustering_results = []

for brand, sample in intent_samples.items():
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=30000
    )

    matrix = vectorizer.fit_transform(sample["text"])
    tfidf_matrices[brand] = matrix

    for k in K_VALUES:
        if matrix.shape[0] <= k:
            continue

        model = KMeans(
            n_clusters=k,
            random_state=SEED,
            n_init=10
        )

        labels = model.fit_predict(matrix)

        score = silhouette_score(
            matrix,
            labels,
            metric="cosine"
        )

        clustering_results.append({
            "brand": brand,
            "k": k,
            "silhouette": score
        })

clustering_results = pd.DataFrame(clustering_results)

display(
    clustering_results.sort_values(
        "silhouette",
        ascending=False
    )
)


In [ ]:
best_cluster_config = (
    clustering_results
    .sort_values("silhouette", ascending=False)
    .groupby("brand")
    .head(1)
    .sort_values("silhouette", ascending=False)
    .reset_index(drop=True)
)

display(best_cluster_config)

best_models = {}
best_labels = {}

for row in best_cluster_config.itertuples(index=False):
    brand = row.brand
    k = int(row.k)

    model = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=10
    )

    labels = model.fit_predict(tfidf_matrices[brand])

    best_models[brand] = model
    best_labels[brand] = labels


In [ ]:
def show_cluster_examples(brand, cluster_id, n_examples=6):
    sample = intent_samples[brand]
    matrix = tfidf_matrices[brand]
    labels = best_labels[brand]
    model = best_models[brand]

    indices = np.where(labels == cluster_id)[0]
    if len(indices) == 0:
        return

    cluster_matrix = matrix[indices]
    centroid = model.cluster_centers_[cluster_id]

    similarities = np.asarray(
        cluster_matrix @ centroid
    ).ravel()

    selected = indices[
        np.argsort(similarities)[::-1][:n_examples]
    ]

    print(f"\n{brand} | Cluster {cluster_id} | {len(indices)} messages")
    print("-" * 90)

    for idx in selected:
        print("•", sample.iloc[idx]["text"])

for brand in [b for b in ["AmazonHelp", "SpotifyCares", "TMobileHelp"]
              if b in best_labels]:
    print("\n" + "#" * 100)
    print("#", brand)
    print("#" * 100)

    sizes = pd.Series(best_labels[brand]).value_counts().sort_index()
    for cluster_id in sizes.index:
        show_cluster_examples(brand, int(cluster_id), n_examples=6)


In [ ]:
cluster_balance = []

for brand, labels in best_labels.items():
    counts = pd.Series(labels).value_counts(normalize=True)

    cluster_balance.append({
        "brand": brand,
        "largest_cluster_share": counts.iloc[0],
        "clusters_ge_5pct": int((counts >= 0.05).sum()),
        "clusters_ge_2pct": int((counts >= 0.02).sum()),
        "effective_cluster_count": 1 / np.sum(counts ** 2)
    })

cluster_balance = pd.DataFrame(cluster_balance)
display(
    cluster_balance.sort_values(
        "effective_cluster_count",
        ascending=False
    )
)


## 7. Historical Response Reuse

A strong candidate should not only have customer messages; it should have historical brand responses that can provide useful evidence for future replies.

We measure response pairs and inspect repeated normalized response patterns.


In [ ]:
pairs_df = pd.DataFrame(brand_customer_pairs)

pairs_df["customer_text"] = pairs_df["customer_tweet_id"].map(tweet_text).fillna("").astype(str)
pairs_df["brand_response"] = pairs_df["brand_tweet_id"].map(tweet_text).fillna("").astype(str)
pairs_df["customer_author"] = pairs_df["customer_tweet_id"].map(tweet_author)

pairs_df["customer_length"] = pairs_df["customer_text"].str.len()
pairs_df["response_length"] = pairs_df["brand_response"].str.len()

def normalize_response(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", "<URL>", text)
    text = re.sub(r"@\w+", "<USER>", text)
    text = re.sub(r"\b\d+\b", "<NUM>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

pairs_df["normalized_response"] = pairs_df["brand_response"].apply(normalize_response)

response_stats = (
    pairs_df.groupby("brand")
    .agg(
        direct_response_pairs=("brand_tweet_id", "count"),
        avg_customer_length=("customer_length", "mean"),
        avg_response_length=("response_length", "mean"),
        unique_customer_messages=("customer_tweet_id", "nunique"),
        unique_brand_responses=("brand_tweet_id", "nunique")
    )
    .reset_index()
)

display(
    response_stats.sort_values(
        "direct_response_pairs",
        ascending=False
    )
)


In [ ]:
template_stats = []

for brand in top_interaction_brands:
    brand_pairs = pairs_df[pairs_df["brand"] == brand]

    if len(brand_pairs) == 0:
        continue

    counts = brand_pairs["normalized_response"].value_counts()
    total = counts.sum()

    template_stats.append({
        "brand": brand,
        "unique_normalized_responses": counts.size,
        "top_1_response_share": counts.iloc[0] / total,
        "top_10_response_share": counts.head(10).sum() / total
    })

template_stats = pd.DataFrame(template_stats)
display(template_stats)


In [ ]:
for brand in [b for b in ["AmazonHelp", "SpotifyCares", "TMobileHelp"]
              if b in pairs_df["brand"].unique()]:

    print("\n" + "=" * 90)
    print(brand)
    print("=" * 90)

    counts = (
        pairs_df[pairs_df["brand"] == brand]["normalized_response"]
        .value_counts()
        .head(10)
    )

    display(counts.to_frame("count"))


## 8. Combine Quantitative Signals

The score below is a **candidate-selection aid**, not a scientific truth. It combines data availability, conversation richness, semantic structure, and response diversity.

The final decision must still include manual inspection of actual conversations, clusters, and historical responses.


In [ ]:
brand_summary = (
    interaction_stats
    .merge(conversation_profiles, on="brand", how="left")
    .merge(cluster_balance, on="brand", how="left")
    .merge(response_stats, on="brand", how="left")
    .merge(
        best_cluster_config[["brand", "k", "silhouette"]].rename(
            columns={"k": "best_k_tfidf", "silhouette": "tfidf_silhouette"}
        ),
        on="brand",
        how="left"
    )
    .merge(template_stats, on="brand", how="left")
)

rank_features = [
    "customer_tweets_with_direct_reply",
    "unique_customers",
    "mean_turns",
    "4plus_turn_rate",
    "6plus_turn_rate",
    "tfidf_silhouette",
    "unique_normalized_responses"
]

for col in rank_features:
    brand_summary[col + "_rank"] = brand_summary[col].rank(pct=True, method="average")

brand_summary["brand_suitability_score"] = (
    0.20 * brand_summary["customer_tweets_with_direct_reply_rank"]
    + 0.10 * brand_summary["unique_customers_rank"]
    + 0.20 * brand_summary["mean_turns_rank"]
    + 0.15 * brand_summary["4plus_turn_rate_rank"]
    + 0.10 * brand_summary["6plus_turn_rate_rank"]
    + 0.15 * brand_summary["tfidf_silhouette_rank"]
    + 0.10 * brand_summary["unique_normalized_responses_rank"]
)

brand_leaderboard = (
    brand_summary
    .sort_values("brand_suitability_score", ascending=False)
    .reset_index(drop=True)
)

leaderboard_columns = [
    "brand",
    "brand_suitability_score",
    "brand_replies",
    "customer_tweets_with_direct_reply",
    "unique_customers",
    "mean_turns",
    "median_turns",
    "4plus_turn_rate",
    "6plus_turn_rate",
    "tfidf_silhouette",
    "unique_normalized_responses"
]

display(
    brand_leaderboard[
        [c for c in leaderboard_columns if c in brand_leaderboard.columns]
    ].head(15)
)


## 9. Finalist Inspection

Do not select the brand only from the numeric score.

For each finalist, inspect:
- conversation depth
- whether customer intents are coherent
- whether historical responses contain actual resolution behavior
- whether the domain depends heavily on unavailable backend state
- whether ambiguous cases exist for a meaningful escalation policy


In [ ]:
FINALISTS = brand_leaderboard.head(5)["brand"].tolist()

print("FINALISTS:")
for i, brand in enumerate(FINALISTS, 1):
    print(f"{i}. {brand}")


In [ ]:
def show_random_conversation(brand, random_state=42):
    customer_ids = list(brand_customer_tweets.get(brand, set()))

    if not customer_ids:
        print("No mapped customer interactions for", brand)
        return

    rng = np.random.default_rng(random_state)
    tweet_id = int(rng.choice(customer_ids))

    chain = get_conversation_chain(tweet_id)

    print("\n" + "=" * 110)
    print("BRAND:", brand)
    print("CONVERSATION LENGTH:", len(chain))
    print("=" * 110)

    for tid in chain:
        role = "CUSTOMER" if inbound_lookup.get(tid) else "BRAND"
        author = author_lookup.get(tid)
        text = tweet_text.get(tid)

        print(f"\n[{role}] {author}")
        print(text)

for brand in FINALISTS:
    print("\n" + "#" * 110)
    print("#", brand)
    print("#" * 110)

    for i in range(3):
        show_random_conversation(brand, random_state=SEED + i)


In [ ]:
manual_criteria = pd.DataFrame({
    "criterion": [
        "Repeated customer problems",
        "Useful historical resolutions",
        "Conversation context is meaningful",
        "Intent taxonomy looks manageable",
        "Enough ambiguous/hard cases",
        "Low dependence on hidden backend state",
        "Good potential for historical-case RAG",
        "Good potential for escalation logic"
    ],
    "score_1_to_5": [None] * 8
})

display(manual_criteria)


## 10. Recommended Decision Rule

Choose the brand that offers the best combination of:

**rich data + meaningful multi-turn conversations + coherent recurring intents + reusable historical resolutions + interesting but evaluable failures.**

Do not automatically choose the largest brand.

For the Hiver submission, document:
1. Why the brand was chosen.
2. Which quantitative signals supported the choice.
3. What manual inspection revealed.
4. What you deliberately did not build because the dataset lacks reliable backend state.


In [ ]:
# Save the main analysis artifacts to /kaggle/working
OUTPUT_DIR = "/kaggle/working/hiver_brand_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

brand_leaderboard.to_csv(f"{OUTPUT_DIR}/brand_leaderboard.csv", index=False)
conversation_profiles.to_csv(f"{OUTPUT_DIR}/conversation_profiles.csv", index=False)
interaction_stats.to_csv(f"{OUTPUT_DIR}/interaction_stats.csv", index=False)
clustering_results.to_csv(f"{OUTPUT_DIR}/clustering_results.csv", index=False)
response_stats.to_csv(f"{OUTPUT_DIR}/response_stats.csv", index=False)
template_stats.to_csv(f"{OUTPUT_DIR}/template_stats.csv", index=False)

# Save a manageable response-pair sample
pairs_df.head(20000).to_csv(
    f"{OUTPUT_DIR}/sample_response_pairs.csv",
    index=False
)

print("Saved:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", name)


# Next Step After Brand Selection

Once the brand is selected, stop doing broad brand analysis.

The next notebook should focus on:

1. Final 8–15 intent taxonomy
2. 150–250-example golden evaluation set
3. Train/validation/test split with leakage controls
4. Majority baseline
5. TF-IDF + Logistic Regression baseline
6. Intent classifier
7. Historical-case retrieval
8. Evidence-grounded response generation
9. Grounding/claim verification
10. Escalation policy
11. LLM-as-judge + human agreement
12. Threshold sweep and ablation studies

The core project thesis:

> **The goal is not maximum automation. It is maximum safe automation.**
